# Azure OpenAI Responses API + File Search Demo

This notebook uses **GPT-5.2** on Azure to answer questions about a document you provide.
Fill in your credentials in **Step 1**, then run all cells top to bottom.

> **Note:** GPT-5.2 requires access approval. Request it at [aka.ms/oai/gpt5access](https://aka.ms/oai/gpt5access).

## Step 0 — Setup

Do these steps **once** in your terminal before opening this notebook.

---

**1. Check that Python is installed**

```bash
python --version
```

You should see `Python 3.11.x` or higher. If not, download it from [python.org](https://www.python.org/downloads/).
On Windows, check **"Add Python to PATH"** during installation.

---

**2. Create a virtual environment**

A virtual environment keeps this project's packages separate from everything else on your machine.

```bash
python -m venv .venv
```

---

**3. Activate the environment**

You'll need to do this every time you open a new terminal for this project.

macOS / Linux:
```bash
source .venv/bin/activate
```

Windows (Command Prompt):
```
.venv\Scripts\activate
```

Your terminal prompt will show `(.venv)` once it's active.

---

**4. Install dependencies**

```bash
pip install -r requirements.txt
```

---

**5. Launch Jupyter**

```bash
jupyter notebook
```

A browser window will open. Click `azure_oai_responses_demo.ipynb` to open this notebook, then run all cells top to bottom.

## Step 1 — Your Azure Credentials

Your credentials are loaded from a **`.env`** file so they stay out of version control.

1. Copy the template:  `cp .env.example .env`
2. Open `.env` and fill in your **Endpoint** and **API Key** (found in the Azure Portal under your Azure OpenAI resource → **Keys and Endpoint**).
3. Set your **deployment name** (found in [Azure AI Foundry](https://oai.azure.com) under **Deployments**).

`REASONING_EFFORT` and `DOCUMENT_PATH` are optional — sensible defaults are used if you leave them blank.

> **Tip:** A sample PDF (`sample_document.pdf`) is included in this repo. To try the PDF upload path, set `DOCUMENT_PATH=sample_document.pdf` in your `.env`. If left blank, a plain-text sample is generated automatically.

In [14]:
from dotenv import dotenv_values
from openai.types.shared import ReasoningEffort

config = dotenv_values(".env")

AZURE_OPENAI_ENDPOINT = config["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_KEY  = config["AZURE_OPENAI_API_KEY"]
MODEL_DEPLOYMENT_NAME = config.get("MODEL_DEPLOYMENT_NAME", "gpt-5.2")

# How deeply GPT-5.2 reasons before answering — "low", "medium", or "high"
REASONING_EFFORT: ReasoningEffort = config.get("REASONING_EFFORT", "medium")  # type: ignore[assignment]

# Path to a document you want to query.
# Leave blank in .env to use the built-in sample, or set to your own file path.
DOCUMENT_PATH: str | None = config.get("DOCUMENT_PATH") or None

## Step 2 — Run the Demo

This cell connects to Azure, loads your document, and asks GPT-5.2 a question about it.

In [15]:
import os
from openai import OpenAI
from openai.types.shared_params import Reasoning
from openai.types.responses import (
    EasyInputMessageParam,
    ResponseInputFileParam,
    ResponseInputTextParam,
)

# --- Connect to Azure ---
client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=f"{AZURE_OPENAI_ENDPOINT.rstrip('/')}/openai/v1/",
)

# --- Prepare document ---
if DOCUMENT_PATH and os.path.exists(DOCUMENT_PATH):
    doc_path = DOCUMENT_PATH
else:
    doc_path = "sample_document.txt"
    with open(doc_path, "w", encoding="utf-8") as f:
        f.write("""\
Contoso CloudSuite — Product FAQ
=================================

PRICING AND PLANS
-----------------
Basic Plan — $29 per user per month
  - Up to 5 users, 25 GB storage per user
  - Email support (48-hour response)
  - 99.5% uptime SLA, no API access

Pro Plan — $79 per user per month
  - Unlimited users, 1 TB storage per user
  - 24/7 phone and email support (4-hour response)
  - 99.9% uptime SLA, full REST API access
  - Dedicated account manager

Enterprise Plan — Custom pricing (contact sales)
  - Everything in Pro, plus on-premises deployment and custom SLA

FEATURES (ALL PLANS)
--------------------
  - Real-time document and spreadsheet collaboration
  - Two-factor authentication, mobile apps, audit logging
  - SSO via SAML 2.0

INTEGRATIONS
------------
Built-in: Slack, Microsoft Teams, Salesforce, GitHub, Jira, Zendesk
Custom integrations via REST API (Pro and Enterprise only)

SECURITY
--------
  - SOC 2 Type II, GDPR, and HIPAA compliant
  - AES-256 encryption at rest, TLS 1.3 in transit
  - Data centers: US-East, EU-West, APAC-Southeast

SUPPORT
-------
Basic:      Email support@contoso.example.com (48-hour response)
Pro:        1-800-555-CLOUD or in-app chat (4-hour response, 24/7)
Enterprise: Dedicated support engineer

CANCELLATION
------------
  - Cancel anytime, no fees; data retained 30 days after cancellation
""")
    print(f"No document specified — created '{doc_path}' as a sample.\n")

# --- Build the document input ---
# Azure's Responses API input_file only supports PDFs.
# For all other file types, the content is read and passed as text.
_, ext = os.path.splitext(doc_path)

doc_input: ResponseInputFileParam | ResponseInputTextParam | None = None

if ext.lower() == ".pdf":
    with open(doc_path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="assistants")
    print(f"Uploaded '{uploaded.filename}' (ID: {uploaded.id})\n")
    doc_input = ResponseInputFileParam(type="input_file", file_id=uploaded.id)
else:
    with open(doc_path, "r", encoding="utf-8") as f:
        doc_input = ResponseInputTextParam(type="input_text", text=f.read())
    print(f"Loaded '{doc_path}'\n")

if doc_input is None:
    raise RuntimeError("No doc input")

# --- Ask a question ---
response = client.responses.create(
    model=MODEL_DEPLOYMENT_NAME,
    instructions="Answer questions based only on the document provided. Be concise.",
    reasoning=Reasoning(effort=REASONING_EFFORT),
    max_output_tokens=4000,
    input=[
        EasyInputMessageParam(
            role="user",
            content=[
                doc_input,
                ResponseInputTextParam(type="input_text", text="Summarize the document, please."),
            ],
        )
    ],
)

print(response.output_text)

Uploaded 'sample_document.pdf' (ID: assistant-RHkCSBmNarFb847iUDuAEs)

**Contoso CloudSuite Summary**

Contoso CloudSuite offers three plans:

- **Basic ($29/user/month):** Up to 5 users, 25 GB storage per user, email support with 48‑hour response, 99.5% uptime, no API access.
- **Pro ($79/user/month):** Unlimited users, 1 TB storage per user, 24/7 phone and email support with 4‑hour response, 99.9% uptime, full REST API access, and a dedicated account manager.
- **Enterprise (custom pricing):** Includes all Pro features plus on‑premises deployment and a custom SLA.

All plans include real-time collaboration, two-factor authentication, SAML-based SSO, mobile apps, and audit logging. The platform integrates with major tools like Slack, Microsoft Teams, Salesforce, GitHub, Jira, and Zendesk, with custom integrations available on Pro and Enterprise plans.

The service is SOC 2 Type II, GDPR, and HIPAA compliant, uses strong encryption, and operates data centers in the US, EU, and APAC reg